In [30]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings("ignore")

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
filepath = '../data/cleanedNairobi_RE_Prices.csv'
df = pd.read_csv(filepath)

In [33]:
df.head()

,Location,Bedroom,bathroom,Price,House_size_sqm,LandSize_acres,propertyType
0,Runda,4.0,4.0,350000000,NaN,0.5,Townhouse
1,Karen,NaN,NaN,30000000,NaN,0.5,Vacant Land
2,Westlands,NaN,NaN,325000000,NaN,0.5,Vacant Land
3,Kitisuru,5.0,5.0,80000000,NaN,0.5,Townhouse
4,Kileleshwa,4.0,4.0,25500000,230.0,NaN,Apartment


##### 🛠️Feature Engineering

###### clean / normalize column names

In [34]:
df.columns = df.columns.str.strip().str.replace(" ", "_").str.lower()

###### Create price per sqm

In [35]:
#with np.errstate(divide='ignore', invalid='ignore'):
  #df['price_per_sqm'] = df['price'] / df['house_size_sqm']


###### Create size buckets

In [36]:
#df['size_category'] = pd.cut(df['house_size_sqm'],
      # bins = [0, 75, 140, 230, 370, np.inf],
       #labels = ['XS', 'S', 'M', 'L', 'XL'])

###### Define Target & Features

In [37]:
target = 'price'
X = df.drop(columns=[target])
y = df[target]

###### Split data

In [38]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

###### Prepocessing

In [39]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.to_list()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.to_list()

In [40]:
# Check missing values %
df.isna().sum() / len(df)

,0
location,0.000000
bedroom,0.102941
bathroom,0.112745
price,0.000000
house_size_sqm,0.514706
landsize_acres,0.686275
propertytype,0.000000


In [41]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [42]:
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

###### -----Model candidates -----

In [44]:
models = {
    'Linear Regrtession': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
}

results = []

# Evaluate each model
for name, model in models.items():
  pipeline = Pipeline([
      ('preprocessor', preprocessor),
      ('regressor', model)
  ])

  pipeline.fit(X_train, y_train)
  y_pred = pipeline.predict(X_test)

  rmse = np.sqrt(mean_squared_error(y_test, y_pred))
  r2 = r2_score(y_test, y_pred)

  results.append({
      'Model': name,
      'RMSE': round(rmse, 2),
      'R2 Score': round(r2, 4)
  })

results_df = pd.DataFrame(results).sort_values(by='RMSE')
print("\n 🔍Model Cmparison:")
print(results_df)


 🔍Model Cmparison:
                Model         RMSE  R2 Score
3             XGBoost  46654464.89    0.9202
2   Gradient Boosting  47369752.06    0.9177
1       Random Forest  63640404.02    0.8515
0  Linear Regrtession  63688652.95    0.8513


###### Feature Importances from Best Model

In [49]:
best_model_name = results_df.iloc[0]['Model']

best_model = models[best_model_name]
best_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', best_model)
])
best_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['bedroom', 'bathroom',
                                                   'house_size_sqm',
                                                   'landsize_acres']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['l...
                              feature_types=None, gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=None,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=None, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=100, n_jobs=None,
                              num_parallel_tree=None, random_state=42, ...))])

In [76]:
if hasattr(best_model, 'feature_importances_'):
  encoded_features = (best_pipeline.named_steps['preprocessor']
                      .transformers_[1][1]
                      .named_steps['onehot']
                      .get_feature_names_out(categorical_features)
                      )
  all_features = np.concatenate([numeric_features, encoded_features])
  importances = best_model.feature_importances_
  feat_imp_df = pd.DataFrame({
      'Feature': all_features,
      'Importance': importances
  }).sort_values(by='Importance', ascending=False).head(10)

  px.bar(feat_imp_df,
         x='Importance',
         y='Feature',
         title=f'Top features in {best_model_name}',
         labels={'Feature': ''}
         ).show()

###### ©️ 2025 Ochieng Fess | Data Strategist